In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Classical feedforward at control flow (dynamic circuits)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Classical feedforward at control flow


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Ang bagong bersyon ng dynamic circuits ay available na ngayon sa lahat ng user sa lahat ng backend. Maaari ka na ngayong mag-run ng dynamic circuits sa utility scale. Tingnan ang [announcement](/announcements/product-updates/2025-09-25-new-dynamic-circuits) para sa karagdagang detalye.

Ang dynamic circuits ay makapangyarihang mga tool na maaari mong gamitin para sukatin ang mga qubit sa gitna ng pagpapatakbo ng quantum circuit at pagkatapos ay magsagawa ng mga classical logic operation sa loob ng circuit, batay sa resulta ng mga mid-circuit na pagsukat. Ang prosesong ito ay kilala rin bilang _classical feedforward_. Bagama't maaga pa lamang ang pag-unawa kung paano pinakamabuti na mapagsamantalahan ang dynamic circuits, ang komunidad ng quantum research ay nakakilala na ng ilang mga use case, tulad ng mga sumusunod:

* Mahusay na paghahanda ng quantum state, tulad ng [GHZ state,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [W-state,](https://arxiv.org/abs/2403.07604) (para sa karagdagang impormasyon tungkol sa W-state, tingnan din ang ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) at isang malawak na klase ng [matrix product states](https://arxiv.org/abs/2404.16083)
* [Mahusay na long-range entanglement](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) sa pagitan ng mga qubit sa parehong chip sa pamamagitan ng paggamit ng mga mababaw na circuit
* Mahusay na [sampling ng IQP-like circuits](https://arxiv.org/pdf/2505.04705)

Ang mga pagpapabuting hatid ng dynamic circuits, gayunpaman, ay may kasamang mga trade-off. Ang mga mid-circuit na pagsukat at classical operations ay karaniwang mas matagal ang oras ng pagpapatupad kaysa sa mga two-qubit gate, at ang pagtaas ng oras na ito ay maaaring mapawalang-bisa ang mga benepisyo ng pinaikling circuit depth. Kaya naman, ang pagpapababa ng haba ng mid-circuit measurement ay isang lugar ng pagpapabuti habang inilalabas ng IBM Quantum&reg; ang [bagong bersyon](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) ng dynamic circuits.

Tinutukoy ng [OpenQASM 3 specification](https://openqasm.com/language/classical.html#looping-and-branching) ang ilang mga control-flow structure, ngunit ang Qiskit Runtime ay kasalukuyang sumusuporta lamang sa conditional na `if` statement. Sa Qiskit SDK, ito ay tumutugma sa pamamaraang [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) sa [QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) Ang pamamaraang ito ay nagbabalik ng isang [context manager](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) at karaniwang ginagamit sa isang `with` statement. Inilalarawan ng gabay na ito kung paano gamitin ang conditional statement na ito.

> **Note:** Ang mga halimbawa ng code sa gabay na ito ay gumagamit ng standard na measure instruction para sa mid-circuit measurements. Gayunpaman, inirerekomenda na gamitin ang [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) instruction sa halip, kung sinusuportahan ng backend. Tingnan ang [dokumentasyon ng Mid-circuit measurements](/guides/measure-qubits#mid-circuit-measurements) para sa mga detalye.
## `if` statement
Ginagamit ang `if` statement para magsagawa ng mga operasyon nang kondisyonal batay sa halaga ng isang classical bit o register.

Sa halimbawa sa ibaba, nag-aaplay kami ng Hadamard gate sa isang qubit at sinusukat ito. Kung ang resulta ay 1, nag-aaplay kami ng X gate sa qubit, na may epektong ibabalik ito sa estado na 0. Pagkatapos ay sinusukat namin ang qubit muli. Ang nagreresultang kinalabasan ng pagsukat ay dapat na 0 na may 100% na posibilidad.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

Ang `with` statement ay maaaring bigyan ng assignment target na isa ring context manager na maaaring i-store at pagkatapos ay gamitin para lumikha ng else block, na isinasagawa tuwing ang mga nilalaman ng `if` block ay *hindi* isinasagawa.

Sa halimbawa sa ibaba, nagpapasimula kami ng mga register na may dalawang qubit at dalawang classical bit. Nag-aaplay kami ng Hadamard gate sa unang qubit at sinusukat ito. Kung ang resulta ay 1, nag-aaplay kami ng Hadamard gate sa pangalawang qubit; kung hindi, nag-aaplay kami ng X gate sa pangalawang qubit. Sa wakas, sinusukat namin ang pangalawang qubit din.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Bukod sa pag-condition sa isang classical bit, posible rin ang pag-condition sa halaga ng isang classical register na binubuo ng maraming bit.

Sa halimbawa sa ibaba, nag-aaplay kami ng Hadamard gate sa dalawang qubit at sinusukat ang mga ito. Kung ang resulta ay `01`, iyon ay, ang unang qubit ay 1 at ang pangalawang qubit ay 0, nag-aaplay kami ng X gate sa ikatlong qubit. Sa wakas, sinusukat namin ang ikatlong qubit. Tandaan na para sa kalinawan, pinili naming tukuyin ang estado ng ikatlong classical bit, na 0, sa kondisyon ng `if`. Sa drawing ng circuit, ang kondisyon ay ipinahiwatig ng mga bilog sa mga classical bit na kina-condition. Ang isang itim na bilog ay nagpapahiwatig ng pag-condition sa 1, habang ang isang puting bilog ay nagpapahiwatig ng pag-condition sa 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Mga classical expression
Ang Qiskit classical expression module na [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) ay naglalaman ng eksplorasyon na representasyon ng mga runtime operation sa mga classical value sa panahon ng pagpapatakbo ng circuit. Dahil sa mga limitasyon ng hardware, ang mga kondisyon ng `QuantumCircuit.if_test()` lamang ang kasalukuyang sinusuportahan.

Ang sumusunod na halimbawa ay nagpapakita na maaari mong gamitin ang pagkalkula ng parity para lumikha ng n-qubit GHZ state gamit ang dynamic circuits. Una, gumawa ng $n/2$ Bell pairs sa mga katabing qubit. Pagkatapos, idikit ang mga pares na ito gamit ang isang layer ng CNOT gate sa pagitan ng mga pares. Pagkatapos ay susukat ng mga target qubit ng lahat ng naunang CNOT gate at iri-reset ang bawat nasukat na qubit sa estado $\vert 0 \rangle$. Mag-aaplay ka ng $X$ sa bawat hindi nasukat na site kung saan ang parity ng lahat ng naunang bit ay kakaiba. Sa wakas, mga CNOT gate ang ilalapat sa mga nasukat na qubit para muling maitatag ang entanglement na nawala sa pagsukat.

Sa pagkalkula ng parity, ang unang elemento ng nabuong expression ay kinabibilangan ng pag-lift ng Python object na `mr[0]` sa isang [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) node (`lift` ay ginagamit para gawing mga classical expression ang mga arbitrary na object). Hindi ito kinakailangan para sa `mr[1]` at sa posibleng susunod na classical register, dahil sila ay mga input sa `expr.bit_xor`, at ang anumang kinakailangang pag-lift ay awtomatikong ginagawa sa mga kasong ito. Ang mga ganitong expression ay maaaring itayo sa mga loop at iba pang mga construct.

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## Hanapin ang mga backend na sumusuporta sa dynamic circuits
Para mahanap ang lahat ng backend na maa-access ng iyong account at sumusuporta sa dynamic circuits, patakbuhin ang code tulad ng sumusunod. Ang halimbawang ito ay ipinapalagay na [na-save mo ang iyong mga login credentials.](/guides/save-credentials) Maaari mo ring [tahasang tukuyin ang mga credentials](/guides/initialize-account#explicit) kapag sinisimulan ang iyong Qiskit Runtime service account. Papayagan ka nitong tingnan ang mga backend na available sa isang partikular na instance o uri ng plano, halimbawa.

> **Note:** - Ang mga backend na available sa account ay depende sa instance na tinukoy sa mga credentials.
> - Ang bagong bersyon ng dynamic circuits ay available na ngayon sa lahat ng user sa lahat ng backend. Tingnan ang [announcement](/announcements/product-updates/2025-09-25-new-dynamic-circuits) para sa karagdagang detalye.